# Notebook 2 — Model Training, Evaluation & XAI
## Diabetic Retinopathy Detection | BRSET Dataset
**Team 24 — Phase 1**  
Study Reference: Tymchenko et al. [5] — EfficientNet-B4 Backbone  
Dataset: Brazilian Multilabel Ophthalmological Dataset (BRSET)

---
### Notebook Structure
```
0. Setup & Configuration
   0.1  Installs & Imports
   0.2  Paths, Constants & Hyperparameters
1. Data Loading
   1.1  Load NB1 Artifacts
   1.2  Train / Val / Test Split Handling
   1.3  Class Balance Summary
2. Dataset & DataLoader
   2.1  BRSETDataset Class
   2.2  Augmentation Transforms (Albumentations)
   2.3  DataLoader Construction
3. Model Architecture
   3.1  EfficientNet-B4 Multilabel Head
   3.2  Parameter Count & Layer Summary
4. Loss Function — Focal Loss with pos_weights
5. Optimizer & Scheduler
6. Training Loop
   6.1  train_epoch / val_epoch Functions
   6.2  Main Training Loop (2-Phase: Frozen → Unfrozen)
7. Evaluation
   7.1  Training Curves
   7.2  Best Model Loading
   7.3  Per-Label Optimal Threshold Search (Val Set)
   7.4  Test Set Metrics (AUC, F1, Precision, Recall, Accuracy)
   7.5  Metric Visualizations
8. Explainable AI (XAI)
   8.1  Grad-CAM — Per-Label Activation Maps
   8.2  SHAP — GradientExplainer Attribution
9. Summary & Export
```

---
**Inputs from Notebook 1:**
- `outputs/labels_preprocessed.csv` — cleaned metadata with preprocessed image paths
- `outputs/preprocessed_images/` — 512×512 preprocessed fundus images
- `outputs/pos_weights.json` — per-label imbalance weights for Focal Loss
- `outputs/pipeline_config.json` — reproducibility config (label cols, image size, seed)

---
## 0. Setup & Configuration

### 0.1 Installs & Imports

In [ ]:
# ── Install required packages ──────────────────────────────────────────────────
# Run once; comment out after environment is set up
!pip install timm albumentations grad-cam shap -q

In [ ]:
# ── Standard library ───────────────────────────────────────────────────────────
import os
import json
import warnings
import random
from pathlib import Path

# ── Scientific stack ───────────────────────────────────────────────────────────
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
import cv2
from PIL import Image
from tqdm.notebook import tqdm

# ── PyTorch ────────────────────────────────────────────────────────────────────
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.cuda.amp import GradScaler, autocast

# ── timm (EfficientNet) ────────────────────────────────────────────────────────
import timm

# ── Albumentations ─────────────────────────────────────────────────────────────
import albumentations as A
from albumentations.pytorch import ToTensorV2

# ── Sklearn metrics ────────────────────────────────────────────────────────────
from sklearn.metrics import (
    roc_auc_score, f1_score, precision_score,
    recall_score, accuracy_score
)
from sklearn.model_selection import train_test_split

# ── XAI ────────────────────────────────────────────────────────────────────────
import shap
from pytorch_grad_cam import GradCAM
from pytorch_grad_cam.utils.image import show_cam_on_image
from pytorch_grad_cam.utils.model_targets import ClassifierOutputTarget

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid', palette='muted', font_scale=1.1)
plt.rcParams.update({'figure.dpi': 120, 'axes.titlesize': 12, 'axes.labelsize': 10})

print('All libraries imported successfully.')

### 0.2 Paths, Constants & Hyperparameters

In [ ]:
# ── Device ─────────────────────────────────────────────────────────────────────
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device : {DEVICE}')
if DEVICE.type == 'cuda':
    print(f'GPU    : {torch.cuda.get_device_name(0)}')
    print(f'VRAM   : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

In [ ]:
# ── Paths ──────────────────────────────────────────────────────────────────────
OUTPUT_DIR = Path('./outputs')          # NB1 artifacts
MODEL_DIR  = Path('./models')           # checkpoints
MODEL_DIR.mkdir(parents=True, exist_ok=True)

LABELS_CSV      = OUTPUT_DIR / 'labels_preprocessed.csv'
POS_WEIGHTS_F   = OUTPUT_DIR / 'pos_weights.json'
PIPELINE_CFG_F  = OUTPUT_DIR / 'pipeline_config.json'

# ── Load NB1 pipeline config ───────────────────────────────────────────────────
with open(PIPELINE_CFG_F) as f:
    pipeline_cfg = json.load(f)

LABEL_COLS = pipeline_cfg['label_columns']
IMG_SIZE   = tuple(pipeline_cfg['target_size'])   # (512, 512) from NB1
SEED       = pipeline_cfg['seed']                 # 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if DEVICE.type == 'cuda':
    torch.cuda.manual_seed_all(SEED)

print(f'Label columns ({len(LABEL_COLS)}): {LABEL_COLS}')
print(f'Image size   : {IMG_SIZE}')
print(f'Seed         : {SEED}')

In [ ]:
# ── Hyperparameters ────────────────────────────────────────────────────────────
# All tunable values in one place — adjust here, not inside cells

# DataLoader
BATCH_SIZE   = 16        # reduce to 8 if OOM on <8 GB VRAM
NUM_WORKERS  = 4

# Training schedule
NUM_EPOCHS      = 40
UNFREEZE_EPOCH  = 5      # Phase 1: train head only for N epochs; then unfreeze backbone
PATIENCE        = 8      # early stopping patience (counted from best val AUC)

# Optimizer
LR_HEAD      = 1e-3      # Phase 1 head-only LR
LR_FULL      = 1e-4      # Phase 2 full-network LR (after unfreeze)
LR_MIN       = 1e-6      # cosine decay floor
WEIGHT_DECAY = 1e-4

# Model
DROPOUT      = 0.3       # classifier head dropout

# Loss
FOCAL_GAMMA       = 2.0  # focal loss focusing parameter
LABEL_SMOOTHING   = 0.05 # soft labels: 0=off, typical range 0.02–0.1

# Evaluation
THRESH_SEARCH_STEP = 0.05   # step size for per-label threshold grid search
DEFAULT_THRESHOLD  = 0.5

# XAI
GRADCAM_N_IMAGES   = 4      # images per label for Grad-CAM grid
SHAP_BACKGROUND_N  = 50     # background images for SHAP (larger = more stable)
SHAP_EXPLAIN_N     = 8      # test images to explain with SHAP

print('Hyperparameters configured.')
print(f'  Batch size      : {BATCH_SIZE}')
print(f'  Epochs          : {NUM_EPOCHS}')
print(f'  Unfreeze after  : {UNFREEZE_EPOCH} epochs')
print(f'  Early stop pat. : {PATIENCE}')
print(f'  LR (head/full)  : {LR_HEAD} / {LR_FULL}')
print(f'  Focal gamma     : {FOCAL_GAMMA}')
print(f'  Label smoothing : {LABEL_SMOOTHING}')

---
## 1. Data Loading

### 1.1 Load NB1 Artifacts

In [ ]:
# ── Load preprocessed label CSV from Notebook 1 ────────────────────────────────
df = pd.read_csv(LABELS_CSV)
print(f'Loaded  : {df.shape[0]} rows × {df.shape[1]} columns')
print(f'Columns : {list(df.columns)}')
df.head(3)

### 1.2 Train / Val / Test Split Handling

In [ ]:
# ── Split handling ─────────────────────────────────────────────────────────────
# If NB1 detected a 'split' column in the original CSV, use it.
# Otherwise fall back to a stratified 70/15/15 random split.

SPLIT_COL = 'split'

if SPLIT_COL in df.columns:
    train_df = df[df[SPLIT_COL] == 'train'].reset_index(drop=True)
    val_df   = df[df[SPLIT_COL] == 'val'].reset_index(drop=True)
    test_df  = df[df[SPLIT_COL] == 'test'].reset_index(drop=True)
    print(f'Using pre-existing split column: "{SPLIT_COL}"')
else:
    # Fallback: patient-aware split to prevent data leakage
    PATIENT_COL = 'patient_id'
    if PATIENT_COL in df.columns:
        unique_patients = df[PATIENT_COL].unique()
        train_pts, tmp_pts = train_test_split(unique_patients, test_size=0.30, random_state=SEED)
        val_pts, test_pts  = train_test_split(tmp_pts, test_size=0.50, random_state=SEED)
        train_df = df[df[PATIENT_COL].isin(train_pts)].reset_index(drop=True)
        val_df   = df[df[PATIENT_COL].isin(val_pts)].reset_index(drop=True)
        test_df  = df[df[PATIENT_COL].isin(test_pts)].reset_index(drop=True)
        print('No split column found — using patient-aware 70/15/15 split (no leakage).')
    else:
        train_df, tmp = train_test_split(df, test_size=0.30, random_state=SEED)
        val_df, test_df = train_test_split(tmp, test_size=0.50, random_state=SEED)
        train_df = train_df.reset_index(drop=True)
        val_df   = val_df.reset_index(drop=True)
        test_df  = test_df.reset_index(drop=True)
        print('No split or patient column found — using random 70/15/15 split.')

print(f'Train : {len(train_df):>5} samples')
print(f'Val   : {len(val_df):>5} samples')
print(f'Test  : {len(test_df):>5} samples')
print(f'Total : {len(train_df) + len(val_df) + len(test_df):>5} samples')

### 1.3 Class Balance Summary

In [ ]:
# ── Per-split positive rate check (leakage / distribution check) ───────────────
balance_rows = []
for split_name, split_df in [('Train', train_df), ('Val', val_df), ('Test', test_df)]:
    pos_rates = split_df[LABEL_COLS].mean().round(4) * 100
    for label, rate in pos_rates.items():
        balance_rows.append({'Split': split_name, 'Label': label, 'Positive Rate (%)': rate})

balance_df = pd.DataFrame(balance_rows)
balance_pivot = balance_df.pivot(index='Label', columns='Split', values='Positive Rate (%)')
print('Positive rate (%) per label per split:')
print(balance_pivot.to_string())

In [ ]:
# ── Load pos_weights from NB1 ──────────────────────────────────────────────────
with open(POS_WEIGHTS_F) as f:
    pos_weight_dict = json.load(f)

# Align to LABEL_COLS order
pos_weights_tensor = torch.tensor(
    [pos_weight_dict[l] for l in LABEL_COLS], dtype=torch.float32
).to(DEVICE)

print('Positive weights loaded (for Focal Loss):')
for label, w in zip(LABEL_COLS, pos_weights_tensor.cpu().numpy()):
    print(f'  {label:<40} {w:.4f}')

---
## 2. Dataset & DataLoader

### 2.1 BRSETDataset Class

In [ ]:
class BRSETDataset(Dataset):
    """
    PyTorch Dataset for the BRSET multilabel fundus image dataset.

    Images are loaded from the preprocessed_path column written by Notebook 1.
    All preprocessing (circle crop, Ben Graham, CLAHE, green normalization,
    512x512 resize) has already been applied and saved to disk by NB1.
    This class only handles augmentation + ImageNet normalization at load time.

    Args:
        df          : DataFrame with 'preprocessed_path' and label columns.
        label_cols  : List of binary label column names.
        transform   : Albumentations transform pipeline.
    """
    def __init__(self, df, label_cols, transform=None):
        self.df        = df.reset_index(drop=True)
        self.label_cols = label_cols
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]

        # Load preprocessed image (BGR from cv2 → convert to RGB)
        img_bgr = cv2.imread(str(row['preprocessed_path']))
        if img_bgr is None:
            # Fallback: black image — should not occur if NB1 ran cleanly
            img_rgb = np.zeros((*IMG_SIZE, 3), dtype=np.uint8)
        else:
            img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)

        # Apply augmentation + normalization
        if self.transform:
            img_rgb = self.transform(image=img_rgb)['image']

        labels = torch.tensor(
            row[self.label_cols].values.astype(np.float32)
        )
        return img_rgb, labels

print('BRSETDataset defined.')

### 2.2 Augmentation Transforms (Albumentations)

In [ ]:
# ── ImageNet normalization statistics ──────────────────────────────────────────
# EfficientNet-B4 is pretrained on ImageNet; use matching mean/std.
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]

# ── Training augmentation pipeline ────────────────────────────────────────────
# Mirrors augmentation strategy in Tymchenko et al. [5] with additions:
# + CoarseDropout (Cutout equivalent, regularizes occlusion sensitivity)
# + ElasticTransform (simulates retinal image device variation)
# + GridDistortion (additional spatial distortion for robustness)
# All parameters are consistent with the preview in Notebook 1.
train_transform = A.Compose([
    A.HorizontalFlip(p=0.5),
    A.VerticalFlip(p=0.3),
    A.Rotate(limit=25, p=0.5, border_mode=cv2.BORDER_REFLECT),
    A.RandomBrightnessContrast(
        brightness_limit=0.2, contrast_limit=0.2, p=0.4
    ),
    A.HueSaturationValue(
        hue_shift_limit=10, sat_shift_limit=15, val_shift_limit=10, p=0.3
    ),
    A.CoarseDropout(
        max_holes=3, max_height=50, max_width=50,
        min_holes=1, min_height=20, min_width=20, p=0.3
    ),
    A.ElasticTransform(
        alpha=120, sigma=6, alpha_affine=3.6, p=0.2,
        border_mode=cv2.BORDER_REFLECT
    ),
    A.GridDistortion(num_steps=5, distort_limit=0.15, p=0.15),
    A.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
    ToTensorV2(),
])

# ── Validation / Test transform (no augmentation) ──────────────────────────────
val_transform = A.Compose([
    A.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
    ToTensorV2(),
])

print('Train and val/test transforms defined.')

### 2.3 DataLoader Construction

In [ ]:
# ── Datasets ───────────────────────────────────────────────────────────────────
train_ds = BRSETDataset(train_df, LABEL_COLS, transform=train_transform)
val_ds   = BRSETDataset(val_df,   LABEL_COLS, transform=val_transform)
test_ds  = BRSETDataset(test_df,  LABEL_COLS, transform=val_transform)

# ── DataLoaders ────────────────────────────────────────────────────────────────
train_loader = DataLoader(
    train_ds, batch_size=BATCH_SIZE, shuffle=True,
    num_workers=NUM_WORKERS, pin_memory=(DEVICE.type == 'cuda'),
    drop_last=True               # keeps batch norm stable on last partial batch
)
val_loader = DataLoader(
    val_ds, batch_size=BATCH_SIZE, shuffle=False,
    num_workers=NUM_WORKERS, pin_memory=(DEVICE.type == 'cuda')
)
test_loader = DataLoader(
    test_ds, batch_size=BATCH_SIZE, shuffle=False,
    num_workers=NUM_WORKERS, pin_memory=(DEVICE.type == 'cuda')
)

print(f'Train  : {len(train_ds):>5} samples | {len(train_loader):>4} batches')
print(f'Val    : {len(val_ds):>5} samples | {len(val_loader):>4} batches')
print(f'Test   : {len(test_ds):>5} samples | {len(test_loader):>4} batches')

In [ ]:
# ── Sanity check: inspect one batch ───────────────────────────────────────────
sample_imgs, sample_labels = next(iter(train_loader))
print(f'Image batch shape  : {sample_imgs.shape}')    # (B, 3, 512, 512)
print(f'Label batch shape  : {sample_labels.shape}')  # (B, num_labels)
print(f'Image dtype        : {sample_imgs.dtype}')
print(f'Label dtype        : {sample_labels.dtype}')
print(f'Pixel value range  : [{sample_imgs.min():.3f}, {sample_imgs.max():.3f}]')

# Visualize one augmented batch
fig, axes = plt.subplots(2, 4, figsize=(16, 8))
axes = axes.flatten()
inv_mean = torch.tensor(IMAGENET_MEAN).view(3, 1, 1)
inv_std  = torch.tensor(IMAGENET_STD).view(3, 1, 1)
for i in range(min(8, BATCH_SIZE)):
    img_display = (sample_imgs[i] * inv_std + inv_mean).clamp(0, 1)
    img_display = img_display.permute(1, 2, 0).numpy()
    label_str = ', '.join(
        [LABEL_COLS[j] for j in range(len(LABEL_COLS)) if sample_labels[i, j] == 1]
    ) or 'negative'
    axes[i].imshow(img_display)
    axes[i].set_title(label_str, fontsize=7, wrap=True)
    axes[i].axis('off')
plt.suptitle('Sample Training Batch (after augmentation + normalization inverse)', fontsize=11, fontweight='bold')
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'eda_plots' / 'sample_train_batch.png', bbox_inches='tight')
plt.show()

---
## 3. Model Architecture

### 3.1 EfficientNet-B4 Multilabel Head

**Design rationale vs Tymchenko et al. [5]:**
- Original paper uses an EfficientNet ensemble (B4 + B5 + SE-ResNeXt50) for 5-class ordinal grading on APTOS.
- We use B4 as the sole backbone — justified as the core model in the paper, and computationally feasible as a single model.
- BRSET is **multilabel binary** (not 5-class ordinal), so we replace the softmax head with a **sigmoid multilabel classifier** and use BCEWithLogits-based Focal Loss.
- We add a **two-layer MLP head** (Linear → BN → SiLU → Dropout → Linear) instead of a single linear projection, which gives the model more capacity to learn per-label decision boundaries.
- Two-phase training: freeze backbone → train head → unfreeze → fine-tune at lower LR.

In [ ]:
class EfficientNetB4Multilabel(nn.Module):
    """
    EfficientNet-B4 backbone with a custom multilabel classification head.

    Architecture:
        Backbone : timm efficientnet_b4 (ImageNet pretrained, head removed)
        Pool     : AdaptiveAvgPool2d → flatten
        Head     : Linear(1792, 512) → BN → SiLU → Dropout → Linear(512, num_classes)

    Output: raw logits (no sigmoid) — consumed by Focal Loss / sigmoid at inference.
    """
    def __init__(self, num_classes, pretrained=True, dropout=DROPOUT):
        super().__init__()
        # Load EfficientNet-B4 without classification head or global pooling
        self.backbone = timm.create_model(
            'efficientnet_b4',
            pretrained=pretrained,
            num_classes=0,        # removes timm's default head
            global_pool=''        # removes timm's global pool; we handle it
        )
        self.pool = nn.AdaptiveAvgPool2d(1)
        in_features = self.backbone.num_features  # 1792 for EfficientNet-B4

        self.classifier = nn.Sequential(
            nn.Dropout(p=dropout),
            nn.Linear(in_features, 512),
            nn.BatchNorm1d(512),
            nn.SiLU(),                        # SiLU = Swish; native to EfficientNet
            nn.Dropout(p=dropout * 0.5),
            nn.Linear(512, num_classes)
        )
        self._init_head()

    def _init_head(self):
        """Xavier uniform init for linear layers in classifier head."""
        for m in self.classifier.modules():
            if isinstance(m, nn.Linear):
                nn.init.xavier_uniform_(m.weight)
                if m.bias is not None:
                    nn.init.zeros_(m.bias)

    def forward(self, x):
        features = self.backbone.forward_features(x)  # (B, 1792, H', W')
        pooled   = self.pool(features).flatten(1)     # (B, 1792)
        return self.classifier(pooled)                # (B, num_classes) — raw logits

    def freeze_backbone(self):
        """Freeze all backbone parameters (Phase 1: head-only training)."""
        for param in self.backbone.parameters():
            param.requires_grad = False
        print('[Phase 1] Backbone frozen — training classifier head only.')

    def unfreeze_backbone(self):
        """Unfreeze all backbone parameters (Phase 2: full fine-tuning)."""
        for param in self.backbone.parameters():
            param.requires_grad = True
        print('[Phase 2] Backbone unfrozen — full network fine-tuning.')


print('EfficientNetB4Multilabel class defined.')

### 3.2 Parameter Count & Layer Summary

In [ ]:
# ── Instantiate model ──────────────────────────────────────────────────────────
model = EfficientNetB4Multilabel(num_classes=len(LABEL_COLS)).to(DEVICE)

total_params     = sum(p.numel() for p in model.parameters())
head_params      = sum(p.numel() for p in model.classifier.parameters())
backbone_params  = sum(p.numel() for p in model.backbone.parameters())

print(f'EfficientNet-B4 loaded (ImageNet pretrained)')
print(f'  Total parameters    : {total_params:>12,}')
print(f'  Backbone parameters : {backbone_params:>12,}')
print(f'  Head parameters     : {head_params:>12,}')
print(f'  Backbone features   : {model.backbone.num_features} (input to head)')
print(f'  Output classes      : {len(LABEL_COLS)}')

# Quick forward pass test
dummy = torch.randn(2, 3, *IMG_SIZE).to(DEVICE)
with torch.no_grad():
    out = model(dummy)
print(f'  Forward pass OK     : input {list(dummy.shape)} → output {list(out.shape)}')
del dummy, out

---
## 4. Loss Function — Focal Loss with pos_weights

In [ ]:
class FocalLoss(nn.Module):
    """
    Multilabel Focal Loss with per-label positive weighting and label smoothing.

    Extends standard BCEWithLogitsLoss with:
    1. Focal weighting: (1 - p_t)^gamma down-weights easy examples.
    2. Positive weights: pos_weight_i = (N - n_i) / n_i addresses class imbalance.
    3. Label smoothing: targets are soft-labeled to prevent overconfidence.

    Args:
        pos_weights     : 1D tensor of shape (num_classes,) from NB1's pos_weights.json.
        gamma           : Focal focusing parameter (2.0 standard; higher → more focus on hard).
        label_smoothing : Smoothing factor ε; target = y*(1-ε) + 0.5*ε.
    """
    def __init__(self, pos_weights, gamma=2.0, label_smoothing=0.05):
        super().__init__()
        self.gamma           = gamma
        self.label_smoothing = label_smoothing
        self.register_buffer('pos_weights', pos_weights)

    def forward(self, logits, targets):
        # Apply label smoothing
        targets_smooth = targets * (1.0 - self.label_smoothing) + 0.5 * self.label_smoothing

        # BCE with logits + positive class weighting
        bce = F.binary_cross_entropy_with_logits(
            logits, targets_smooth,
            pos_weight=self.pos_weights,
            reduction='none'
        )
        # Compute p_t for focal weighting
        probs = torch.sigmoid(logits)
        p_t   = probs * targets + (1.0 - probs) * (1.0 - targets)
        focal_weight = (1.0 - p_t) ** self.gamma

        return (focal_weight * bce).mean()


criterion = FocalLoss(
    pos_weights=pos_weights_tensor,
    gamma=FOCAL_GAMMA,
    label_smoothing=LABEL_SMOOTHING
)
print(f'FocalLoss initialized (gamma={FOCAL_GAMMA}, label_smoothing={LABEL_SMOOTHING})')

---
## 5. Optimizer & Scheduler

**Two-phase training strategy:**
- **Phase 1 (epochs 1–`UNFREEZE_EPOCH`):** Backbone frozen. Only the classifier head is trained at `LR_HEAD`. This warms up the randomly initialized head without corrupting pretrained features.
- **Phase 2 (epoch `UNFREEZE_EPOCH+1` onward):** Backbone unfrozen. Full network fine-tuned at `LR_FULL` (10× lower) with CosineAnnealingLR decaying to `LR_MIN`.
- **Gradient clipping:** max norm = 1.0 prevents exploding gradients after unfreeze.
- **Mixed precision (AMP):** GradScaler + autocast halves memory use and speeds up training.

In [ ]:
# ── Phase 1 optimizer (head only) ─────────────────────────────────────────────
model.freeze_backbone()

optimizer = optim.AdamW(
    filter(lambda p: p.requires_grad, model.parameters()),
    lr=LR_HEAD,
    weight_decay=WEIGHT_DECAY
)
scheduler = optim.lr_scheduler.CosineAnnealingLR(
    optimizer, T_max=UNFREEZE_EPOCH, eta_min=LR_MIN
)
scaler = GradScaler()  # mixed precision scaler

trainable_now = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Phase 1 — Trainable params: {trainable_now:,} (head only)')
print(f'Phase 1 — LR: {LR_HEAD}, Weight decay: {WEIGHT_DECAY}')

---
## 6. Training Loop

### 6.1 train_epoch / val_epoch Functions

In [ ]:
def train_epoch(model, loader, criterion, optimizer, scaler, device):
    """
    One training epoch with mixed precision and gradient clipping.
    Returns: (mean_loss, macro_auc)
    """
    model.train()
    total_loss = 0.0
    all_probs, all_targets = [], []

    for imgs, labels in tqdm(loader, desc='  Train', leave=False):
        imgs, labels = imgs.to(device), labels.to(device)

        optimizer.zero_grad(set_to_none=True)

        with autocast():
            logits = model(imgs)
            loss   = criterion(logits, labels)

        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        scaler.step(optimizer)
        scaler.update()

        total_loss += loss.item()
        all_probs.append(torch.sigmoid(logits).detach().cpu().numpy())
        all_targets.append(labels.detach().cpu().numpy())

    all_probs   = np.concatenate(all_probs)
    all_targets = np.concatenate(all_targets)

    try:
        macro_auc = roc_auc_score(all_targets, all_probs, average='macro')
    except ValueError:
        macro_auc = 0.0   # raised if a label has only one class in batch

    return total_loss / len(loader), macro_auc


def val_epoch(model, loader, criterion, device):
    """
    Validation epoch — no gradient computation.
    Returns: (mean_loss, macro_auc, all_probs, all_targets)
    """
    model.eval()
    total_loss = 0.0
    all_probs, all_targets = [], []

    with torch.no_grad():
        for imgs, labels in tqdm(loader, desc='  Val  ', leave=False):
            imgs, labels = imgs.to(device), labels.to(device)
            with autocast():
                logits = model(imgs)
                loss   = criterion(logits, labels)
            total_loss += loss.item()
            all_probs.append(torch.sigmoid(logits).cpu().numpy())
            all_targets.append(labels.cpu().numpy())

    all_probs   = np.concatenate(all_probs)
    all_targets = np.concatenate(all_targets)

    try:
        macro_auc = roc_auc_score(all_targets, all_probs, average='macro')
    except ValueError:
        macro_auc = 0.0

    return total_loss / len(loader), macro_auc, all_probs, all_targets


print('train_epoch() and val_epoch() defined.')

### 6.2 Main Training Loop (2-Phase: Frozen → Unfrozen)

In [ ]:
# ── Training state ─────────────────────────────────────────────────────────────
history = {
    'train_loss': [], 'val_loss': [],
    'train_auc':  [], 'val_auc':  [],
    'lr':         []
}
best_val_auc     = 0.0
patience_counter = 0
best_model_path  = MODEL_DIR / 'efficientnet_b4_best.pth'
phase            = 1

print('Starting training...')
print(f'Phase 1: epochs 1–{UNFREEZE_EPOCH} (head only, LR={LR_HEAD})')
print(f'Phase 2: epochs {UNFREEZE_EPOCH+1}–{NUM_EPOCHS} (full network, LR={LR_FULL})')
print('-' * 75)

for epoch in range(1, NUM_EPOCHS + 1):

    # ── Phase transition at UNFREEZE_EPOCH ──────────────────────────────────
    if epoch == UNFREEZE_EPOCH + 1 and phase == 1:
        phase = 2
        model.unfreeze_backbone()
        optimizer = optim.AdamW(
            model.parameters(), lr=LR_FULL, weight_decay=WEIGHT_DECAY
        )
        scheduler = optim.lr_scheduler.CosineAnnealingLR(
            optimizer,
            T_max=(NUM_EPOCHS - UNFREEZE_EPOCH),
            eta_min=LR_MIN
        )
        scaler = GradScaler()   # reset scaler for new optimizer
        trainable_now = sum(p.numel() for p in model.parameters() if p.requires_grad)
        print(f'  → Phase 2 started. Trainable params: {trainable_now:,}\n')

    # ── Train & validate ─────────────────────────────────────────────────────
    train_loss, train_auc = train_epoch(
        model, train_loader, criterion, optimizer, scaler, DEVICE
    )
    val_loss, val_auc, val_probs, val_targets = val_epoch(
        model, val_loader, criterion, DEVICE
    )
    scheduler.step()

    lr_now = optimizer.param_groups[0]['lr']

    # ── Log ──────────────────────────────────────────────────────────────────
    history['train_loss'].append(train_loss)
    history['val_loss'].append(val_loss)
    history['train_auc'].append(train_auc)
    history['val_auc'].append(val_auc)
    history['lr'].append(lr_now)

    status = ''
    if val_auc > best_val_auc:
        best_val_auc = val_auc
        patience_counter = 0
        torch.save({
            'epoch':              epoch,
            'model_state_dict':   model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'val_auc':            val_auc,
            'val_loss':           val_loss,
            'label_cols':         LABEL_COLS,
            'history':            history,
        }, best_model_path)
        status = '  ✓ BEST'
    else:
        patience_counter += 1

    print(
        f'Ep {epoch:03d}/{NUM_EPOCHS} | Ph{phase} | '
        f'TrLoss {train_loss:.4f} TrAUC {train_auc:.4f} | '
        f'VaLoss {val_loss:.4f} VaAUC {val_auc:.4f} | '
        f'LR {lr_now:.2e}{status}'
    )

    if patience_counter >= PATIENCE:
        print(f'\n[Early Stop] No val AUC improvement for {PATIENCE} consecutive epochs.')
        break

print('-' * 75)
print(f'Training complete. Best Val AUC: {best_val_auc:.4f}')
print(f'Best model saved to: {best_model_path}')

---
## 7. Evaluation

### 7.1 Training Curves

In [ ]:
# ── Training history plots ─────────────────────────────────────────────────────
epochs_ran = list(range(1, len(history['train_loss']) + 1))

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Loss
axes[0].plot(epochs_ran, history['train_loss'], color='steelblue', label='Train', linewidth=1.5)
axes[0].plot(epochs_ran, history['val_loss'],   color='salmon',    label='Val',   linewidth=1.5)
axes[0].axvline(UNFREEZE_EPOCH, color='grey', linestyle='--', linewidth=1, label=f'Unfreeze @ ep{UNFREEZE_EPOCH}')
axes[0].set_title('Focal Loss')
axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('Loss')
axes[0].legend(); axes[0].grid(True, alpha=0.3)

# Macro AUC
axes[1].plot(epochs_ran, history['train_auc'], color='steelblue', label='Train', linewidth=1.5)
axes[1].plot(epochs_ran, history['val_auc'],   color='salmon',    label='Val',   linewidth=1.5)
axes[1].axvline(UNFREEZE_EPOCH, color='grey', linestyle='--', linewidth=1)
axes[1].axhline(0.9, color='green', linestyle=':', linewidth=1, alpha=0.7, label='AUC=0.90 target')
axes[1].set_title('Macro AUC-ROC')
axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('AUC')
axes[1].legend(); axes[1].grid(True, alpha=0.3)

# Learning rate
axes[2].semilogy(epochs_ran, history['lr'], color='mediumorchid', linewidth=1.5)
axes[2].axvline(UNFREEZE_EPOCH, color='grey', linestyle='--', linewidth=1)
axes[2].set_title('Learning Rate Schedule')
axes[2].set_xlabel('Epoch'); axes[2].set_ylabel('LR (log scale)')
axes[2].grid(True, alpha=0.3)

plt.suptitle('EfficientNet-B4 — Training History (BRSET)', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'eda_plots' / 'training_curves.png', bbox_inches='tight')
plt.show()
print(f'Best Val AUC: {best_val_auc:.4f} | Epochs trained: {len(epochs_ran)}')

### 7.2 Best Model Loading

In [ ]:
# ── Load best checkpoint ───────────────────────────────────────────────────────
checkpoint = torch.load(best_model_path, map_location=DEVICE)
model.load_state_dict(checkpoint['model_state_dict'])
model.eval()

print(f'Best checkpoint loaded:')
print(f'  Epoch     : {checkpoint["epoch"]}')
print(f'  Val AUC   : {checkpoint["val_auc"]:.4f}')
print(f'  Val Loss  : {checkpoint["val_loss"]:.4f}')

In [ ]:
# ── Collect predictions on val and test sets ───────────────────────────────────
def collect_predictions(model, loader, device):
    """Run inference and collect sigmoid probabilities and ground-truth targets."""
    model.eval()
    all_probs, all_targets = [], []
    with torch.no_grad():
        for imgs, labels in tqdm(loader, desc='  Inference', leave=False):
            imgs = imgs.to(device)
            with autocast():
                logits = model(imgs)
            all_probs.append(torch.sigmoid(logits).cpu().numpy())
            all_targets.append(labels.numpy())
    return np.concatenate(all_probs), np.concatenate(all_targets)


print('Collecting val set predictions...')
val_probs, val_targets_np = collect_predictions(model, val_loader, DEVICE)

print('Collecting test set predictions...')
test_probs, test_targets_np = collect_predictions(model, test_loader, DEVICE)

print(f'Val  probs shape : {val_probs.shape}')
print(f'Test probs shape : {test_probs.shape}')

### 7.3 Per-Label Optimal Threshold Search (Val Set)

For multilabel classification, a single 0.5 threshold is suboptimal because each label has a different prevalence (shown in the EDA). We search for the threshold that maximizes F1 on the **validation set** per label, then apply these thresholds to the test set — no test-set information leaks into threshold selection.

In [ ]:
# ── Per-label optimal threshold search (val set F1 maximization) ───────────────
thresh_candidates = np.arange(0.10, 0.85, THRESH_SEARCH_STEP)
best_thresholds = {}

print(f'Searching thresholds in [{thresh_candidates[0]:.2f}, {thresh_candidates[-1]:.2f}] step {THRESH_SEARCH_STEP}...')
print(f'{"Label":<40} {"Threshold":>10} {"Val F1":>10}')
print('-' * 62)

for i, label in enumerate(LABEL_COLS):
    best_thresh, best_f1 = DEFAULT_THRESHOLD, 0.0
    for t in thresh_candidates:
        preds_bin = (val_probs[:, i] >= t).astype(int)
        f1 = f1_score(val_targets_np[:, i], preds_bin, zero_division=0)
        if f1 > best_f1:
            best_f1   = f1
            best_thresh = t
    best_thresholds[label] = round(float(best_thresh), 4)
    print(f'{label:<40} {best_thresh:>10.2f} {best_f1:>10.4f}')

# Save thresholds
with open(OUTPUT_DIR / 'per_label_thresholds.json', 'w') as f:
    json.dump(best_thresholds, f, indent=2)
print('\nThresholds saved to outputs/per_label_thresholds.json')

### 7.4 Test Set Metrics (AUC, F1, Precision, Recall, Accuracy)

In [ ]:
# ── Per-label test set evaluation ─────────────────────────────────────────────
results = []
for i, label in enumerate(LABEL_COLS):
    thresh    = best_thresholds[label]
    preds_bin = (test_probs[:, i] >= thresh).astype(int)
    true_bin  = test_targets_np[:, i].astype(int)

    try:
        auc = roc_auc_score(true_bin, test_probs[:, i])
    except ValueError:
        auc = float('nan')  # only one class present in test set for this label

    results.append({
        'Label':     label,
        'AUC-ROC':   round(auc, 4) if not np.isnan(auc) else np.nan,
        'F1':        round(f1_score(true_bin, preds_bin, zero_division=0), 4),
        'Precision': round(precision_score(true_bin, preds_bin, zero_division=0), 4),
        'Recall':    round(recall_score(true_bin, preds_bin, zero_division=0), 4),
        'Accuracy':  round(accuracy_score(true_bin, preds_bin), 4),
        'Threshold': thresh,
        'Support':   int(true_bin.sum()),
    })

results_df = pd.DataFrame(results)
print('=' * 85)
print('  PER-LABEL TEST SET PERFORMANCE — EfficientNet-B4 (BRSET)')
print('=' * 85)
print(results_df.to_string(index=False))
results_df.to_csv(OUTPUT_DIR / 'per_label_metrics.csv', index=False)

In [ ]:
# ── Macro / micro aggregate metrics ───────────────────────────────────────────
valid_mask = ~results_df['AUC-ROC'].isna()

macro_auc       = results_df.loc[valid_mask, 'AUC-ROC'].mean()
macro_f1        = results_df['F1'].mean()
macro_precision = results_df['Precision'].mean()
macro_recall    = results_df['Recall'].mean()
macro_accuracy  = results_df['Accuracy'].mean()

# Micro: flatten all labels
all_preds_flat   = np.concatenate([
    (test_probs[:, i] >= best_thresholds[l]).astype(int)
    for i, l in enumerate(LABEL_COLS)
])
all_targets_flat = test_targets_np.T.flatten()

micro_f1        = f1_score(all_targets_flat, all_preds_flat, average='micro', zero_division=0)
micro_recall    = recall_score(all_targets_flat, all_preds_flat, average='micro', zero_division=0)
micro_precision = precision_score(all_targets_flat, all_preds_flat, average='micro', zero_division=0)

print('\n' + '=' * 50)
print('  AGGREGATE METRICS — TEST SET')
print('=' * 50)
print(f'  Macro AUC-ROC   : {macro_auc:.4f}')
print(f'  Macro F1        : {macro_f1:.4f}')
print(f'  Macro Precision : {macro_precision:.4f}')
print(f'  Macro Recall    : {macro_recall:.4f}')
print(f'  Macro Accuracy  : {macro_accuracy:.4f}')
print(f'  Micro F1        : {micro_f1:.4f}')
print(f'  Micro Recall    : {micro_recall:.4f}')
print(f'  Micro Precision : {micro_precision:.4f}')
print('=' * 50)

### 7.5 Metric Visualizations

In [ ]:
# ── Per-label metric bar charts ────────────────────────────────────────────────
metrics_to_plot = ['AUC-ROC', 'F1', 'Precision', 'Recall', 'Accuracy']
n_metrics = len(metrics_to_plot)
n_cols = 3
n_rows = (n_metrics + n_cols - 1) // n_cols

fig, axes = plt.subplots(n_rows, n_cols, figsize=(n_cols * 6, n_rows * 4))
axes = axes.flatten()
palette = sns.color_palette('Set2', len(LABEL_COLS))

for idx, metric in enumerate(metrics_to_plot):
    ax = axes[idx]
    vals = results_df[metric].fillna(0)
    bars = ax.barh(results_df['Label'], vals, color=palette, edgecolor='white')
    ax.set_xlim(0, 1.12)
    ax.axvline(0.90, color='green', linestyle='--', linewidth=1, alpha=0.6, label='0.90')
    ax.set_title(f'{metric} per Label', fontweight='bold')
    ax.legend(fontsize=8)
    for bar, v in zip(bars, vals):
        ax.text(v + 0.01, bar.get_y() + bar.get_height() / 2,
                f'{v:.3f}', va='center', fontsize=8)

for j in range(idx + 1, len(axes)):
    axes[j].set_visible(False)

plt.suptitle('EfficientNet-B4 — Per-Label Performance (Test Set)',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'eda_plots' / 'per_label_metrics.png', bbox_inches='tight')
plt.show()

In [ ]:
# ── ROC curves per label ───────────────────────────────────────────────────────
from sklearn.metrics import roc_curve

n_labels = len(LABEL_COLS)
n_cols_roc = min(4, n_labels)
n_rows_roc = (n_labels + n_cols_roc - 1) // n_cols_roc

fig, axes = plt.subplots(n_rows_roc, n_cols_roc,
                          figsize=(n_cols_roc * 4, n_rows_roc * 3.5))
axes = axes.flatten() if n_labels > 1 else [axes]

for i, label in enumerate(LABEL_COLS):
    true_bin = test_targets_np[:, i].astype(int)
    if true_bin.sum() == 0 or (1 - true_bin).sum() == 0:
        axes[i].text(0.5, 0.5, 'Single class\nin test set',
                     ha='center', va='center', transform=axes[i].transAxes)
        axes[i].set_title(label, fontsize=8)
        continue
    fpr, tpr, _ = roc_curve(true_bin, test_probs[:, i])
    auc_val = results_df.loc[results_df['Label'] == label, 'AUC-ROC'].values[0]
    axes[i].plot(fpr, tpr, color='steelblue', linewidth=1.5)
    axes[i].plot([0, 1], [0, 1], 'k--', linewidth=0.8)
    axes[i].fill_between(fpr, tpr, alpha=0.1, color='steelblue')
    axes[i].set_title(f'{label}\nAUC={auc_val:.3f}', fontsize=8, fontweight='bold')
    axes[i].set_xlabel('FPR', fontsize=7); axes[i].set_ylabel('TPR', fontsize=7)
    axes[i].tick_params(labelsize=7)

for j in range(i + 1, len(axes)):
    axes[j].set_visible(False)

plt.suptitle('ROC Curves per Label — EfficientNet-B4 (Test Set)',
             fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'eda_plots' / 'roc_curves_per_label.png', bbox_inches='tight')
plt.show()

In [ ]:
# ── Precision-Recall curves per label ─────────────────────────────────────────
from sklearn.metrics import precision_recall_curve, average_precision_score

fig, axes = plt.subplots(n_rows_roc, n_cols_roc,
                          figsize=(n_cols_roc * 4, n_rows_roc * 3.5))
axes = axes.flatten() if n_labels > 1 else [axes]

for i, label in enumerate(LABEL_COLS):
    true_bin = test_targets_np[:, i].astype(int)
    if true_bin.sum() == 0:
        axes[i].text(0.5, 0.5, 'No positives', ha='center', va='center',
                     transform=axes[i].transAxes)
        axes[i].set_title(label, fontsize=8)
        continue
    prec, rec, _ = precision_recall_curve(true_bin, test_probs[:, i])
    ap = average_precision_score(true_bin, test_probs[:, i])
    axes[i].plot(rec, prec, color='salmon', linewidth=1.5)
    axes[i].fill_between(rec, prec, alpha=0.1, color='salmon')
    axes[i].set_title(f'{label}\nAP={ap:.3f}', fontsize=8, fontweight='bold')
    axes[i].set_xlabel('Recall', fontsize=7); axes[i].set_ylabel('Precision', fontsize=7)
    axes[i].tick_params(labelsize=7)

for j in range(i + 1, len(axes)):
    axes[j].set_visible(False)

plt.suptitle('Precision-Recall Curves per Label — EfficientNet-B4 (Test Set)',
             fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'eda_plots' / 'pr_curves_per_label.png', bbox_inches='tight')
plt.show()

---
## 8. Explainable AI (XAI)

Both Grad-CAM and SHAP are applied here, after training, using the best checkpoint. They require the trained model weights, so they have no meaningful place in an earlier notebook.

| Method | What it shows | Scope |
|--------|--------------|-------|
| **Grad-CAM** | Which spatial regions of the retinal image drive each label prediction | Per-image, per-label |
| **SHAP GradientExplainer** | Pixel-level attribution averaged over a test cohort | Aggregate + per-image |

Together they provide both instance-level (Grad-CAM) and population-level (SHAP aggregate) interpretability, consistent with Zhang et al. [14].

### 8.1 Grad-CAM — Per-Label Activation Maps

In [ ]:
# ── Grad-CAM setup ─────────────────────────────────────────────────────────────
# Target layer: last convolutional block of EfficientNet-B4 backbone.
# model.backbone.blocks[-1] is the final MBConv block before the conv_head.
# This is the standard target for EfficientNet Grad-CAM in the literature.

target_layers = [model.backbone.blocks[-1]]
cam = GradCAM(
    model=model,
    target_layers=target_layers,
    use_cuda=(DEVICE.type == 'cuda')
)
print(f'GradCAM initialized on: model.backbone.blocks[-1]')
print(f'Target layer type: {type(target_layers[0]).__name__}')

In [ ]:
# ── Grad-CAM grid: one positive sample per label ───────────────────────────────
# For each label, find a test set image that is truly positive (ground truth = 1)
# and visualize the model's activation map for that label.

def get_gradcam_sample(label, label_idx, test_df, model, cam, val_transform, IMG_SIZE, device):
    """Compute Grad-CAM overlay for the first positive test sample of a label."""
    pos_rows = test_df[test_df[label] == 1]
    if len(pos_rows) == 0:
        return None

    row = pos_rows.iloc[0]
    img_bgr = cv2.imread(str(row['preprocessed_path']))
    if img_bgr is None:
        return None
    img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)

    # Model input
    img_tensor = val_transform(image=img_rgb)['image'].unsqueeze(0).to(device)

    # Grad-CAM
    targets        = [ClassifierOutputTarget(label_idx)]
    grayscale_cam  = cam(input_tensor=img_tensor, targets=targets)[0]  # (H, W)

    # Overlay
    img_float  = img_rgb.astype(np.float32) / 255.0
    cam_overlay = show_cam_on_image(img_float, grayscale_cam, use_rgb=True)

    return img_rgb, grayscale_cam, cam_overlay


# Collect results
gradcam_results = []
for i, label in enumerate(LABEL_COLS):
    result = get_gradcam_sample(label, i, test_df, model, cam, val_transform, IMG_SIZE, DEVICE)
    if result is not None:
        gradcam_results.append((label, *result))
    else:
        print(f'  Skipping "{label}" — no positive samples in test set.')

print(f'Grad-CAM computed for {len(gradcam_results)} labels.')

In [ ]:
# ── Plot Grad-CAM grid ─────────────────────────────────────────────────────────
n_results = len(gradcam_results)
fig, axes = plt.subplots(n_results, 3, figsize=(12, n_results * 3.2))
if n_results == 1:
    axes = axes[np.newaxis, :]

col_titles = ['Original (preprocessed)', 'CAM Heatmap', 'CAM Overlay']
for col_i, title in enumerate(col_titles):
    axes[0, col_i].set_title(title, fontsize=10, fontweight='bold')

for row_i, (label, img_rgb, grayscale_cam, cam_overlay) in enumerate(gradcam_results):
    axes[row_i, 0].imshow(img_rgb)
    axes[row_i, 0].set_ylabel(label, fontsize=8, fontweight='bold', rotation=0,
                               labelpad=120, va='center')
    axes[row_i, 0].axis('off')

    axes[row_i, 1].imshow(grayscale_cam, cmap='jet')
    axes[row_i, 1].axis('off')

    axes[row_i, 2].imshow(cam_overlay)
    axes[row_i, 2].axis('off')

plt.suptitle(
    'Grad-CAM Activation Maps — EfficientNet-B4\n(one positive test sample per label)',
    fontsize=12, fontweight='bold'
)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'eda_plots' / 'gradcam_per_label.png',
            bbox_inches='tight', dpi=120)
plt.show()
print('Grad-CAM grid saved.')

In [ ]:
# ── Multi-sample Grad-CAM for the primary DR label ─────────────────────────────
# Show GRADCAM_N_IMAGES positive samples for the diabetic retinopathy label
# to illustrate model consistency across different fundus images.

# Detect the primary DR label — prefer exact match, fall back to substring
DR_LABEL_CANDIDATES = [l for l in LABEL_COLS if 'diabetic_retinopathy' in l.lower()]
if not DR_LABEL_CANDIDATES:
    DR_LABEL_CANDIDATES = [l for l in LABEL_COLS if 'retinopathy' in l.lower() or 'dr' in l.lower()]
DR_LABEL = DR_LABEL_CANDIDATES[0] if DR_LABEL_CANDIDATES else LABEL_COLS[0]
DR_IDX   = LABEL_COLS.index(DR_LABEL)

print(f'Primary DR label for multi-sample Grad-CAM: "{DR_LABEL}" (index {DR_IDX})')

dr_pos_rows = test_df[test_df[DR_LABEL] == 1].head(GRADCAM_N_IMAGES)
if len(dr_pos_rows) == 0:
    print('No positive DR samples in test set — skipping multi-sample Grad-CAM.')
else:
    fig, axes = plt.subplots(len(dr_pos_rows), 3,
                              figsize=(12, len(dr_pos_rows) * 3))
    if len(dr_pos_rows) == 1:
        axes = axes[np.newaxis, :]

    for col_i, title in enumerate(col_titles):
        axes[0, col_i].set_title(title, fontsize=9, fontweight='bold')

    for row_i, (_, row) in enumerate(dr_pos_rows.iterrows()):
        img_bgr = cv2.imread(str(row['preprocessed_path']))
        img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
        img_tensor = val_transform(image=img_rgb)['image'].unsqueeze(0).to(DEVICE)
        grayscale_cam = cam(
            input_tensor=img_tensor, targets=[ClassifierOutputTarget(DR_IDX)]
        )[0]
        cam_overlay = show_cam_on_image(img_rgb.astype(np.float32) / 255.0,
                                         grayscale_cam, use_rgb=True)
        axes[row_i, 0].imshow(img_rgb);          axes[row_i, 0].axis('off')
        axes[row_i, 1].imshow(grayscale_cam, cmap='jet'); axes[row_i, 1].axis('off')
        axes[row_i, 2].imshow(cam_overlay);      axes[row_i, 2].axis('off')

    plt.suptitle(f'Grad-CAM — Multi-sample view for "{DR_LABEL}"',
                 fontsize=12, fontweight='bold')
    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / 'eda_plots' / f'gradcam_{DR_LABEL}_multisample.png',
                bbox_inches='tight', dpi=120)
    plt.show()

### 8.2 SHAP — GradientExplainer Attribution

`shap.GradientExplainer` is the correct choice for CNN-based models:
- It uses integrated gradients over a background distribution — more theoretically grounded than GradCAM for attributions.
- `DeepExplainer` is faster but unreliable with BatchNorm and SiLU activations (common in EfficientNet).
- We limit to `SHAP_BACKGROUND_N` background samples and `SHAP_EXPLAIN_N` test images to keep compute tractable.

In [ ]:
# ── Collect background images (train set sample) ───────────────────────────────
print(f'Collecting {SHAP_BACKGROUND_N} background images from train set...')
background_list = []
collected = 0
for imgs, _ in train_loader:
    background_list.append(imgs)
    collected += imgs.shape[0]
    if collected >= SHAP_BACKGROUND_N:
        break

background = torch.cat(background_list)[:SHAP_BACKGROUND_N].to(DEVICE)
print(f'Background tensor shape: {background.shape}')

# ── Collect explain images (test set) ──────────────────────────────────────────
print(f'Collecting {SHAP_EXPLAIN_N} test images for SHAP explanation...')
explain_list = []
collected = 0
for imgs, _ in test_loader:
    explain_list.append(imgs)
    collected += imgs.shape[0]
    if collected >= SHAP_EXPLAIN_N:
        break

explain = torch.cat(explain_list)[:SHAP_EXPLAIN_N].to(DEVICE)
print(f'Explain tensor shape: {explain.shape}')

In [ ]:
# ── Compute SHAP values ────────────────────────────────────────────────────────
# GradientExplainer wraps the full model.
# shap_values is a list of length num_classes,
# each element is shape (N_explain, C, H, W).
model.eval()
print('Initializing SHAP GradientExplainer...')
explainer = shap.GradientExplainer(model, background)

print(f'Computing SHAP values for {SHAP_EXPLAIN_N} test images × {len(LABEL_COLS)} labels...')
print('(This may take several minutes on CPU; ~1–2 min on GPU)')
shap_values = explainer.shap_values(explain)  # list[num_labels] of (N, C, H, W)

print(f'SHAP values computed.')
print(f'  Type        : {type(shap_values)}')
print(f'  Num outputs : {len(shap_values)}')
print(f'  Shape each  : {shap_values[0].shape}')  # (N, 3, 512, 512)

In [ ]:
# ── SHAP: mean absolute attribution map per label ──────────────────────────────
# Collapse (N, 3, H, W) → (H, W) by averaging over samples and color channels.
# This gives a spatial map of which image regions are most influential per label,
# averaged across the explain cohort.

n_labels  = len(LABEL_COLS)
n_cols_sh = min(4, n_labels)
n_rows_sh = (n_labels + n_cols_sh - 1) // n_cols_sh

fig, axes = plt.subplots(n_rows_sh, n_cols_sh,
                          figsize=(n_cols_sh * 4, n_rows_sh * 4))
axes = axes.flatten() if n_labels > 1 else [axes]

for i, label in enumerate(LABEL_COLS):
    # mean |SHAP| over N samples and 3 channels → (H, W)
    shap_map = np.abs(shap_values[i]).mean(axis=(0, 1))  # mean over N, C
    im = axes[i].imshow(shap_map, cmap='hot', interpolation='bilinear')
    axes[i].set_title(f'{label}', fontsize=8, fontweight='bold')
    axes[i].axis('off')
    plt.colorbar(im, ax=axes[i], fraction=0.046, pad=0.04)

for j in range(i + 1, len(axes)):
    axes[j].set_visible(False)

plt.suptitle(
    'SHAP GradientExplainer — Mean |Attribution| per Label\n'
    f'(averaged over {SHAP_EXPLAIN_N} test images)',
    fontsize=12, fontweight='bold'
)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'eda_plots' / 'shap_mean_attribution_per_label.png',
            bbox_inches='tight', dpi=120)
plt.show()
print('SHAP aggregate attribution map saved.')

In [ ]:
# ── SHAP: per-image explanation for primary DR label ──────────────────────────
# Show individual SHAP attribution overlaid on the original images.
# Red = positive contribution (pushes prediction toward positive class).
# Blue = negative contribution.

SHAP_SHOW_N = min(4, SHAP_EXPLAIN_N)

# Unnormalize for display
inv_mean = np.array(IMAGENET_MEAN).reshape(3, 1, 1)
inv_std  = np.array(IMAGENET_STD).reshape(3, 1, 1)

fig, axes = plt.subplots(SHAP_SHOW_N, 3,
                          figsize=(12, SHAP_SHOW_N * 3))
if SHAP_SHOW_N == 1:
    axes = axes[np.newaxis, :]

axes[0, 0].set_title('Original Image', fontsize=9, fontweight='bold')
axes[0, 1].set_title(f'SHAP Attribution\n("{DR_LABEL}")', fontsize=9, fontweight='bold')
axes[0, 2].set_title('SHAP Overlay', fontsize=9, fontweight='bold')

for img_i in range(SHAP_SHOW_N):
    # Unnormalized image
    img_np = (explain[img_i].cpu().numpy() * inv_std + inv_mean).clip(0, 1)  # (3, H, W)
    img_display = img_np.transpose(1, 2, 0)  # (H, W, 3)

    # SHAP map for DR label, this image — mean over color channels
    shap_map = shap_values[DR_IDX][img_i].mean(axis=0)  # (H, W)

    # Normalize SHAP for display (diverging: blue=neg, red=pos)
    abs_max = np.abs(shap_map).max() + 1e-8
    shap_norm = shap_map / abs_max  # [-1, 1]

    # Overlay: blend SHAP heatmap onto image
    # Positive SHAP → red channel emphasis; negative → blue
    overlay = img_display.copy()
    pos_mask = (shap_norm > 0.1)
    neg_mask = (shap_norm < -0.1)
    overlay[pos_mask, 0] = np.clip(overlay[pos_mask, 0] + shap_norm[pos_mask] * 0.6, 0, 1)
    overlay[neg_mask, 2] = np.clip(overlay[neg_mask, 2] - shap_norm[neg_mask] * 0.6, 0, 1)

    axes[img_i, 0].imshow(img_display)
    axes[img_i, 0].axis('off')

    im = axes[img_i, 1].imshow(shap_norm, cmap='RdBu_r', vmin=-1, vmax=1)
    axes[img_i, 1].axis('off')
    plt.colorbar(im, ax=axes[img_i, 1], fraction=0.046, pad=0.04)

    axes[img_i, 2].imshow(overlay)
    axes[img_i, 2].axis('off')

plt.suptitle(
    f'SHAP Per-Image Attribution — "{DR_LABEL}"\n'
    'Red = positive attribution | Blue = negative attribution',
    fontsize=11, fontweight='bold'
)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'eda_plots' / 'shap_per_image_dr_label.png',
            bbox_inches='tight', dpi=120)
plt.show()

In [ ]:
# ── Side-by-side: Grad-CAM vs SHAP for the DR label ──────────────────────────
# Direct comparison of the two XAI methods on the same image.

dr_pos_test = test_df[test_df[DR_LABEL] == 1]
if len(dr_pos_test) == 0:
    print('No positive DR samples in test set — skipping comparison.')
else:
    sample_row = dr_pos_test.iloc[0]
    img_bgr    = cv2.imread(str(sample_row['preprocessed_path']))
    img_rgb    = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)

    # Grad-CAM
    img_tensor = val_transform(image=img_rgb)['image'].unsqueeze(0).to(DEVICE)
    grayscale_cam = cam(
        input_tensor=img_tensor, targets=[ClassifierOutputTarget(DR_IDX)]
    )[0]
    gradcam_overlay = show_cam_on_image(
        img_rgb.astype(np.float32) / 255.0, grayscale_cam, use_rgb=True
    )

    # SHAP (first explain image — approximate)
    shap_map_compare = shap_values[DR_IDX][0].mean(axis=0)
    abs_max_c = np.abs(shap_map_compare).max() + 1e-8

    fig, axes = plt.subplots(1, 4, figsize=(16, 4))
    axes[0].imshow(img_rgb)
    axes[0].set_title('Original', fontweight='bold')
    axes[0].axis('off')

    axes[1].imshow(grayscale_cam, cmap='jet')
    axes[1].set_title('Grad-CAM Heatmap', fontweight='bold')
    axes[1].axis('off')

    axes[2].imshow(gradcam_overlay)
    axes[2].set_title('Grad-CAM Overlay', fontweight='bold')
    axes[2].axis('off')

    im = axes[3].imshow(shap_map_compare / abs_max_c, cmap='RdBu_r', vmin=-1, vmax=1)
    axes[3].set_title('SHAP Attribution', fontweight='bold')
    axes[3].axis('off')
    plt.colorbar(im, ax=axes[3], fraction=0.046, pad=0.04)

    plt.suptitle(
        f'XAI Comparison — Grad-CAM vs SHAP | Label: "{DR_LABEL}"',
        fontsize=12, fontweight='bold'
    )
    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / 'eda_plots' / 'xai_gradcam_vs_shap_comparison.png',
                bbox_inches='tight', dpi=120)
    plt.show()
    print('Grad-CAM vs SHAP comparison saved.')

---
## 9. Summary & Export

In [ ]:
# ── Save final metrics summary as JSON ─────────────────────────────────────────
summary = {
    'model': 'EfficientNet-B4 (timm, ImageNet pretrained)',
    'task': f'Multilabel classification ({len(LABEL_COLS)} labels)',
    'dataset': 'BRSET (Brazilian Multilabel Ophthalmological Dataset)',
    'reference_study': 'Tymchenko et al. [5]',
    'hyperparameters': {
        'batch_size':       BATCH_SIZE,
        'epochs_trained':   len(history['train_loss']),
        'unfreeze_epoch':   UNFREEZE_EPOCH,
        'lr_head':          LR_HEAD,
        'lr_full':          LR_FULL,
        'weight_decay':     WEIGHT_DECAY,
        'focal_gamma':      FOCAL_GAMMA,
        'label_smoothing':  LABEL_SMOOTHING,
        'dropout':          DROPOUT,
    },
    'test_set_metrics': {
        'macro_auc_roc':   round(float(macro_auc),       4),
        'macro_f1':        round(float(macro_f1),        4),
        'macro_precision': round(float(macro_precision), 4),
        'macro_recall':    round(float(macro_recall),    4),
        'macro_accuracy':  round(float(macro_accuracy),  4),
        'micro_f1':        round(float(micro_f1),        4),
        'micro_recall':    round(float(micro_recall),    4),
    },
    'best_val_auc': round(float(best_val_auc), 4),
    'per_label_thresholds': best_thresholds,
    'per_label_metrics': results_df.to_dict(orient='records'),
}

with open(OUTPUT_DIR / 'model_summary.json', 'w') as f:
    json.dump(summary, f, indent=2)

print('model_summary.json saved.')

In [ ]:
# ── Final printed report ───────────────────────────────────────────────────────
print('=' * 70)
print('  NOTEBOOK 2 — FINAL SUMMARY')
print('=' * 70)
print(f'  Model          : EfficientNet-B4 (timm, ImageNet pretrained)')
print(f'  Task           : Multilabel ({len(LABEL_COLS)} labels, sigmoid + Focal Loss)')
print(f'  Epochs trained : {len(history["train_loss"])} / {NUM_EPOCHS}')
print(f'  Best Val AUC   : {best_val_auc:.4f}')
print()
print('  Test Set Performance:')
print(f'    Macro AUC-ROC   : {macro_auc:.4f}')
print(f'    Macro F1        : {macro_f1:.4f}')
print(f'    Macro Recall    : {macro_recall:.4f}')
print(f'    Macro Precision : {macro_precision:.4f}')
print(f'    Macro Accuracy  : {macro_accuracy:.4f}')
print(f'    Micro F1        : {micro_f1:.4f}')
print()
print('  XAI Methods Applied:')
print(f'    Grad-CAM  : model.backbone.blocks[-1] | per-label + multi-sample')
print(f'    SHAP      : GradientExplainer | background={SHAP_BACKGROUND_N} | explain={SHAP_EXPLAIN_N}')
print()
print('  Saved artifacts:')
print(f'    models/efficientnet_b4_best.pth')
print(f'    outputs/model_summary.json')
print(f'    outputs/per_label_metrics.csv')
print(f'    outputs/per_label_thresholds.json')
print(f'    outputs/eda_plots/training_curves.png')
print(f'    outputs/eda_plots/per_label_metrics.png')
print(f'    outputs/eda_plots/roc_curves_per_label.png')
print(f'    outputs/eda_plots/pr_curves_per_label.png')
print(f'    outputs/eda_plots/gradcam_per_label.png')
print(f'    outputs/eda_plots/gradcam_{DR_LABEL}_multisample.png')
print(f'    outputs/eda_plots/shap_mean_attribution_per_label.png')
print(f'    outputs/eda_plots/shap_per_image_dr_label.png')
print(f'    outputs/eda_plots/xai_gradcam_vs_shap_comparison.png')
print('=' * 70)